# MediAssist AI

## Objective

Build a Healthcare Q&A System that:

- Retrieves Intermittent Fasting research articles from PubMed
- Stores article embeddings in ChromaDB
- Retrieves relevant evidence using vector search
- Supports LLM-powered question answering through RAG

### Technologies

- PubMed API
- ChromaDB
- Sentence Transformers
- LLaMA
- Retrieval Augmented Generation (RAG)

In [30]:
import os
import sys

sys.path.append("../src")

## Step 1: Retrieve Articles from PubMed

Goal:

- Search PubMed for Intermittent Fasting literature
- Retrieve up to 300 PMIDs
- Fetch article metadata and abstracts
- Prepare data for vector storage

In [6]:
from pubmed import PubMedRetriever

In [7]:
search_term = """
(intermittent fasting) AND
(obesity OR type 2 diabetes OR metabolic syndrome)
"""

In [11]:
pmids = PubMedRetriever.search_pubmed_articles(
    search_term=search_term,
    max_results=300
)

print(len(pmids))

300


In [12]:
articles = PubMedRetriever.fetch_pubmed_abstracts(pmids)

print(len(articles))

[{'pmid': '42227869', 'title': 'Feasibility and preliminary evaluation of a lacto-vegetarian diet combined with intermittent fasting in women with type 2 diabetes: A pilot randomized controlled trial.', 'abstract': {'SUMMARY': 'BackgroundWomen living with type 2 diabetes mellitus (T2DM) often encounter unique physiological and lifestyle-related hurdles that impact disease progression and management. Dietary interventions tailored to cultural and gender-specific needs are essential for effective diabetes management.AimThis pilot trial primarily evaluated feasibility and safety and, secondarily, explored the preliminary metabolic effects and practical implementation of a lacto-vegetarian diet combined with intermittent fasting in Indian women with T2DM.MethodsIn this 12-week, parallel-group, open-label randomized controlled trial, 10 women with T2DM (aged 25-60 years) were randomly allocated in a 1:1 ratio using block randomization to either an intervention group (lacto-vegetarian diet w

In [15]:
import pandas as pd
len(articles)
pd.DataFrame(articles).shape

(300, 6)

In [16]:
df = pd.DataFrame(articles)

df.to_json(
    "../data/pubmed_articles.json",
    orient="records",
    indent=4
)

In [17]:
print(len(pmids))
print(len(articles))

300
300


## Step 2: Create ChromaDB Vector Collection

In this step we:

- Create a persistent Chroma database
- Convert PubMed articles into searchable documents
- Generate embeddings
- Store documents in a vector collection
- Enable semantic retrieval

In [18]:
from chroma_manager import ChromaManager

In [19]:
vector_db = ChromaManager()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
vector_db.add_documents(articles)

300 documents added


In [21]:
vector_db.count()

300

### Chroma Collection Created

Collection Name:
pubmed_articles

Embedding Model:
all-MiniLM-L6-v2

Documents Stored:
300 PubMed Articles

## Step 3: Semantic Retrieval Testing

This step validates that ChromaDB can retrieve
relevant PubMed articles based on natural language queries.

In [22]:
results = vector_db.retrieve(
    query="How does intermittent fasting affect type 2 diabetes?",
    n_results=5
)

results

{'ids': [['41503866', '40367729', '40416375', '41527226', '39926793']],
 'embeddings': None,
 'documents': [['\n            Title: [Clinical considerations regarding the effect of intermittent fasting in people diagnosed with type 2 diabetes].\n\n            Abstract:\n            No Abstract\n            ',
   "\n            Title: The metabolic effects of intermittent fasting in patients with type 2 diabetes exist in the short term but disappear after its discontinuation: A systematic review and meta-analysis of randomized controlled trials.\n\n            Abstract:\n            This meta-analysis aimed to determine the short- (< 3 months) and long-term (≥ 3 months) metabolic effects of IF in patients with type 2 diabetes. We hypothesized that IF is non-inferior to other dietary control methods (including continuous energy restriction, standard diet, Mediterranean diet and ad libitum diet) in terms of both short-term and long-term metabolic impacts in patients with type 2 diabetes. W

In [23]:
for i, doc in enumerate(results["documents"][0]):

    print("=" * 80)
    print(f"RESULT {i+1}")
    print("=" * 80)

    print(doc[:1000])
    print()

RESULT 1

            Title: [Clinical considerations regarding the effect of intermittent fasting in people diagnosed with type 2 diabetes].

            Abstract:
            No Abstract
            

RESULT 2

            Title: The metabolic effects of intermittent fasting in patients with type 2 diabetes exist in the short term but disappear after its discontinuation: A systematic review and meta-analysis of randomized controlled trials.

            Abstract:
            This meta-analysis aimed to determine the short- (< 3 months) and long-term (≥ 3 months) metabolic effects of IF in patients with type 2 diabetes. We hypothesized that IF is non-inferior to other dietary control methods (including continuous energy restriction, standard diet, Mediterranean diet and ad libitum diet) in terms of both short-term and long-term metabolic impacts in patients with type 2 diabetes. We searched for studies in the MEDLINE, EMBASE, and Cochrane Library until August 20, 2023. Studies with no

In [24]:
results = vector_db.retrieve(
    query="Effects of intermittent fasting on obesity",
    n_results=5
)

for doc in results["documents"][0]:
    print(doc[:800])
    print("\n")


            Title: Is intermittent fasting an effective intervention for adults living with obesity?

            Abstract:
            No Abstract
            



            Title: Metabolic and Neuroendocrine Responses to Intermittent Fasting in Obesity.

            Abstract:
            
            



            Title: In overweight or obesity, intermittent fasting does not differ from dietary advice but increases weight loss vs. no intervention.

            Abstract:
            GIM/FP/GP: [Formula: see text] Public Health: [Formula: see text].
            



            Title: Intermittent fasting: Evidence for benefit, lack of effect, or potential cardiometabolic risk?

            Abstract:
            No Abstract
            



            Title: Intermittent fasting for adults with overweight or obesity.

            Abstract:
            Weight loss remains the primary strategy for reducing health risks and societal consequences associated with overweight and obesity

In [25]:
results = vector_db.retrieve(
    query="Can intermittent fasting improve insulin sensitivity?",
    n_results=5
)

for doc in results["documents"][0]:
    print(doc[:800])
    print("\n")


            Title: [Clinical considerations regarding the effect of intermittent fasting in people diagnosed with type 2 diabetes].

            Abstract:
            No Abstract
            



            Title: Is intermittent fasting an effective intervention for adults living with obesity?

            Abstract:
            No Abstract
            



            Title: Intermittent fasting as a promising strategy for the treatment of metabolic syndrome: all that glitters ain't gold.

            Abstract:
            No Abstract
            



            Title: Insulin resistance reduction, intermittent fasting, and human growth hormone: secondary analysis of a randomized trial.

            Abstract:
            Intense intermittent fasting regimens safely reduce weight to a similar extent as caloric restriction. A previous trial reported low-frequency 26-week intermittent fasting reduced homeostatic model assessment of insulin resistance (HOMA-IR) without significant weight 

In [26]:
results["metadatas"][0]

[{'journal': 'Nutricion hospitalaria', 'publication_date': '2026'},
 {'publication_date': '2026', 'journal': 'Journal of primary health care'},
 {'publication_date': '2025', 'journal': 'Evidence-based nursing'},
 {'journal': 'npj metabolic health and disease', 'publication_date': '2024'},
 {'journal': 'Journal of pharmacy & bioallied sciences',
  'publication_date': '2024'}]

### Retrieval Validation

The vector database successfully retrieves
scientifically relevant PubMed articles using
semantic similarity search.

This validates the retrieval component required
for the RAG pipeline.

In [27]:
filtered_articles = []

for article in articles:

    abstract_text = " ".join(
        article["abstract"].values()
    )

    if (
        abstract_text.strip()
        and abstract_text != "No Abstract"
    ):
        filtered_articles.append(article)

print(len(filtered_articles))

266


In [28]:
vector_db = ChromaManager(
    collection_name="pubmed_articles_filtered"
)

In [29]:
vector_db.add_documents(filtered_articles)

266 documents added


## Step 4: Retrieval-Augmented Generation Preparation

The RAG pipeline retrieves the most relevant
PubMed articles and combines them into a
single context block for LLM-based answer generation.

In [4]:
from rag_pipeline import RAGPipeline

In [5]:
rag = RAGPipeline()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
context = rag.retrieve_context(
    "How effective is intermittent fasting for type 2 diabetes?"
)

print(context[:3000])


            Title: Effect of intermittent fasting on diabetic patients-A narrative review.

            Abstract:
            Managing diabetes requires careful food choices, especially for those with limited access to nutritious options. Intermittent fasting (IF) has emerged as a promising strategy for improving outcomes in both type 1 (T1DM) and type 2 diabetes mellitus (T2DM) by stabilizing blood glucose levels and aiding in weight management. This review explores various methods of IF, including the 16/8 method, 5:2 diet, Eat-Stop-Eat, and others, highlighting their potential benefits such as weight loss, improved insulin sensitivity, and glucose tolerance. Although IF shows promise, particularly in T2DM, it poses risks like hypoglycemia and dehydration, particularly in T1DM. Safe practices include consulting healthcare providers, monitoring glucose and ketone levels, and adjusting medications. This review highlights the need for individualized approaches to IF to optimize diabete

## Step 5: Retrieval-Augmented Generation

In [6]:
from rag_pipeline import RAGPipeline

rag = RAGPipeline()

In [7]:
response = rag.answer_question(
    "How effective is intermittent fasting for Type 2 Diabetes?"
)

print(response)

**Short Answer:** Intermittent fasting (IF) has been shown to be effective in improving glycemic control and weight loss in patients with type 2 diabetes mellitus (T2DM). However, the duration of these benefits is uncertain.

**Key Findings:**

1. A systematic review and meta-analysis found that IF significantly decreased glycated hemoglobin A1c (HbA1c), fasting plasma glucose (FPG), and body weight in the short term (< 3 months) compared to control interventions. However, these metabolic benefits did not persist after discontinuation of IF.
2. Another study found that IF resulted in improved glycaemic control, insulin sensitivity, facilitated adherence to recommendations, weight reduction, and lower risk of complications in patients with T2DM.
3. A systematic review and meta-analysis comparing the effects of IF in insulin-treated versus oral hypoglycemic agents (OHAs) treated patients with T2DM found that IF was an effective adjuvant therapeutic strategy for improving glycemic control

In [8]:
rag.answer_question(
    "Can intermittent fasting help obesity?"
)

'**Short Answer:**\nYes, intermittent fasting may be a helpful approach for weight loss in individuals with obesity. While it may not differ significantly from dietary advice, it can lead to greater weight loss and improved health outcomes compared to no intervention or waiting list.\n\n**Key Findings:**\n\n1. Compared to regular dietary advice, intermittent fasting may result in little to no difference in percentage weight loss from baseline (MD -0.33, 95% CI -0.92 to 0.26; 21 studies, 1430 participants; low-certainty evidence due to risk of bias).\n2. Intermittent fasting may have little to no effect on achieving a 5% reduction in body weight (RR 0.98, 95% CI 0.82 to 1.18; 4 studies, 472 participants; very low-certainty evidence due to risk of bias and imprecision).\n3. Compared to no intervention or waiting list, intermittent fasting likely results in little to no difference in percentage weight loss from baseline (MD -3.42, 95% CI -4.95 to -1.90; 6 studies, 427 participants; modera

In [9]:
rag.answer_question(
    "What are the risks of intermittent fasting in diabetic patients?"
)

'**Short Answer:**\nThe risks of intermittent fasting (IF) in diabetic patients are hypoglycemia and dehydration, particularly in type 1 diabetes. Safe practices include consulting healthcare providers, monitoring glucose and ketone levels, and adjusting medications.\n\n**Key Findings:**\n\n* A meta-analysis of randomized controlled trials found that IF significantly decreased glycated hemoglobin A1c (HbA1c), fasting plasma glucose (FPG), and body weight in the short term compared to control interventions. However, these benefits disappeared after discontinuation of IF. (Source: "The metabolic effects of intermittent fasting in patients with type 2 diabetes exist in the short term but disappear after its discontinuation: A systematic review and meta-analysis of randomized controlled trials.")\n* Another study found that a 16:8 intermittent fasting regimen significantly reduced fasting blood glucose levels and anthropometric indices in obese type 2 diabetes patients over four weeks, wit

In [10]:
rag.answer_question(
    "Which intermittent fasting method is most effective?"
)

'**Short Answer:**\nThe evidence suggests that different intermittent fasting methods may not differ significantly in terms of weight loss or other health outcomes. However, some studies suggest that intermittent fasting may be more effective than regular dietary advice for weight loss, particularly when compared to no intervention or waiting list.\n\n**Key Findings:**\n\n1. A Cochrane review (2024) including 22 studies found that:\n\t* Intermittent fasting may result in little to no difference in percentage weight loss from baseline compared to regular dietary advice.\n\t* Intermittent fasting may have little to no effect on achieving a 5% reduction in body weight, but the evidence is very uncertain.\n2. A study (2023) comparing intermittent fasting with non-fasting found that:\n\t* Fasters had fewer depressive symptoms, but stress levels were similar between the two groups.\n\t* The diet of fasters contained lower levels of proteins, lipids, and carbohydrates, while non-fasters had a

In [21]:
import importlib
import rag_pipeline

importlib.reload(rag_pipeline)

from rag_pipeline import RAGPipeline

rag = RAGPipeline()

In [22]:
response = rag.answer_question(
    "How effective is intermittent fasting for Type 2 Diabetes?"
)

print(response)

## Short Answer

Intermittent fasting (IF) has been found to be effective in improving outcomes for patients with type 2 diabetes mellitus (T2DM). IF has been shown to reduce HbA1c, fasting plasma glucose, and body weight in the short-term compared to control interventions. However, the metabolic benefits of IF may not endure after its discontinuation.

## Evidence Summary

* A narrative review found that various methods of IF, including 16/8 method, 5:2 diet, Eat-Stop-Eat, and others, can improve outcomes for patients with T1DM and T2DM by stabilizing blood glucose levels and aiding in weight management.
* A systematic review and meta-analysis found that IF significantly decreased HbA1c, FPG, and body weight in the short-term compared to control interventions. However, the metabolic benefits of IF did not endure after its discontinuation.

## Clinical Considerations

* Patients with type 2 diabetes should consult their healthcare provider before starting an intermittent fasting regime

In [23]:
from rag_pipeline_groq import RAGPipeline

In [24]:
rag = RAGPipeline()

In [25]:
response = rag.answer_question(
    "How effective is intermittent fasting for Type 2 Diabetes?"
)

print(response)

**1. Short Answer**
Intermittent fasting (IF) is effective in the short term for Type 2 Diabetes, improving glycemic control, insulin sensitivity, and weight reduction. However, its long-term benefits may not persist after discontinuation, suggesting the need for continual IF practice.

**2. Evidence Summary**
The evidence from the provided studies suggests that IF can significantly decrease glycated hemoglobin A1c (HbA1c), fasting plasma glucose (FPG), and body weight in the short term (< 3 months) compared to control interventions. A meta-analysis of 12 articles with 966 participants found that IF decreased HbA1c (SMD: -0.93), FPG (SMD: -0.73), and body weight (SMD: -1.11) in the short term. However, the long-term benefits (> 3 months) of IF were similar to control interventions, indicating that the metabolic benefits of IF may not endure after its discontinuation.

**3. Clinical Considerations**
When implementing IF for Type 2 Diabetes management, clinicians should consider the foll

In [36]:
from article_search import search_articles

results = search_articles(
    "intermittent fasting"
)

print(len(results))
print(results[0]["title"])

20
Feasibility and preliminary evaluation of a lacto-vegetarian diet combined with intermittent fasting in women with type 2 diabetes: A pilot randomized controlled trial.
